# Agent 2 — Security Collection and Validation with NVD + GitHub

Notebook goals:

- a **real collection-and-validation workflow**, not a chatbot,
- **NVD** as the main structured vulnerability source,
- **GitHub** as optional enrichment and as a recovery path when NVD coverage is weak,
- **explicit state**,
- **explicit routing**,
- **plain Python node functions**,
- **visible loops and clear stop conditions**.

> The model interprets the task into **bounded semantic fields**.  
> Python then builds the actual NVD and GitHub queries, validates them, and controls the loop.

That gives the model more semantic control without letting it generate unstable raw API requests.


## Local setup

This notebook is self-contained.
Uncomment the install line if you need it.


In [1]:

%pip install -q langgraph langchain langchain-ollama pydantic requests langsmith python-dotenv


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:

from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Literal, Optional, TypedDict

import requests
from pydantic import BaseModel, Field

from langgraph.graph import START, END, StateGraph
from langgraph.checkpoint.memory import InMemorySaver
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGSMITH_PROJECT"] = "workshop_02"

## Runtime flags

A few small switches make the notebook easier to teach and easier to debug.


In [ ]:

USE_REAL_LLM = True
USE_REAL_TOOLS = True
USE_STRUCTURED_OUTPUT = True
USE_CHECKPOINTER = False

ENABLE_GITHUB_ENRICHMENT = True
ENABLE_TASK_GITHUB_RECOVERY = True

OLLAMA_MODEL = "gemma4:e4b"

NVD_RESULTS_PER_PAGE = 5
GITHUB_RESULTS_PER_PAGE = 3


USER_AGENT = "agent-workshop-notebook/1.0"

TARGET_STRONG_MATCHES = 2
TARGET_PLATFORM_STRONG_MATCHES = 1
MAX_NVD_QUERIES = 6
MAX_GITHUB_RECOVERY_QUERIES = 4


## LLM helper functions

We keep the model boundary small and visible.

Important choices in this notebook:

1. We use **`SystemMessage`** and **`HumanMessage`** directly.
2. The model returns a **bounded semantic plan**, not raw API parameters.
3. Python owns the final request construction and validation.


In [ ]:

llm = ChatOllama(
    model=OLLAMA_MODEL,
    temperature=0,
    num_ctx=10000,
    reasoning=True,
)

def call_llm_text(messages: list) -> str:
    response = llm.invoke(messages)
    return response.content if hasattr(response, "content") else str(response)

def call_llm_structured(messages: list, schema: type[BaseModel]):
    structured_llm = llm.with_structured_output(schema)
    result = structured_llm.invoke(messages)
    if result is None:
        raise ValueError("Structured output call returned None.")
    return result

checkpointer = InMemorySaver() if USE_CHECKPOINTER else None


## Structured models

We use Pydantic where it improves clarity:

- the planner output,
- stored candidate records,
- validation decisions,
- enrichment records,
- extracted signals.

We do **not** force the whole graph state into Pydantic.
The shared state stays explicit and readable as a `TypedDict`.


In [ ]:

class QueryPlan(BaseModel):
    cve_id: Optional[str] = None
    required_concepts: list[str] = Field(default_factory=list, min_length=1, max_length=3)
    required_platforms: list[str] = Field(default_factory=list, max_length=3)
    optional_evidence_terms: list[str] = Field(default_factory=list, max_length=4)
    github_terms: list[str] = Field(default_factory=list, max_length=5)
    strict_platform: bool = True
    reasoning: str

class CandidateRecord(BaseModel):
    source: Literal["nvd", "github"]
    item_id: str
    url: Optional[str] = None
    text_preview: str
    accepted: bool
    partial_match: bool = False
    score: int = 0
    matched_concepts: list[str] = Field(default_factory=list)
    matched_platforms: list[str] = Field(default_factory=list)
    reason: str

class ValidationDecision(BaseModel):
    accepted: bool
    partial_match: bool
    score: int
    matched_concepts: list[str] = Field(default_factory=list)
    matched_platforms: list[str] = Field(default_factory=list)
    reason: str

class EnrichmentRecord(BaseModel):
    related_to_item_id: str
    query: str
    repo_name: str
    repo_url: str
    text_preview: str

class ExtractedSignals(BaseModel):
    source: str
    item_id: str
    matched_terms: list[str]
    platform_terms_found: list[str]
    mentions_exploit: bool
    mentions_remote_code_execution: bool
    mentions_prompt_injection: bool
    mentions_data_exfiltration: bool
    mentions_privilege_escalation: bool
    severity_hint: Optional[str] = None


## Graph state

In [ ]:

class CollectorState(TypedDict):
    task: str
    interpreted_plan: Optional[dict]

    search_phase: Literal["nvd", "github_recovery"]

    nvd_query_candidates: list[dict]
    nvd_query_index: int
    current_nvd_query: Optional[dict]

    github_recovery_queries: list[str]
    github_recovery_query_index: int
    current_github_recovery_query: Optional[str]

    candidate_items: list[dict]
    current_candidate_index: int
    current_candidate: Optional[dict]

    accepted_items: list[dict]
    partial_items: list[dict]
    rejected_items: list[dict]
    github_enrichments: list[dict]
    extracted_signals: list[dict]

    queries_tried: list[str]
    run_log: list[str]
    errors: list[str]

    iteration: int
    max_iterations: int

    accepted_count_at_last_assessment: int
    no_new_accept_rounds: int
    max_stale_rounds: int

    last_validation_accepted: bool
    last_validation_partial: bool
    last_validation_source: Optional[str]

    github_recovery_attempted: bool
    stop_reason: Optional[str]
    summary: Optional[str]


## Semantic planning helpers

This is the key design boundary.

The **model interprets the task semantically**.
Python then:

- removes generic filler,
- sanitizes phrases,
- expands platform aliases,
- builds strong **conjunctive** NVD queries first,
- builds conservative GitHub queries,
- and keeps the request shape stable.

That is safer and easier to debug than letting the model emit raw API syntax.


In [ ]:
CVE_RE = re.compile(r"\bCVE-\d{4}-\d{4,}\b", re.IGNORECASE)

SAFE_TERM_RE = re.compile(r"^[A-Za-z0-9][A-Za-z0-9 ._\-/]{0,79}$")
SAFE_GITHUB_QUERY_RE = re.compile(r'^[A-Za-z0-9" :,_\-./]{1,120}$')

PLATFORM_ALIASES = {
    "macos": ["macos", "os x", "osx", "mac os"],
    "ios": ["ios", "iphone", "ipad"],
}

COMMON_PLATFORM_HINTS = {"macos", "windows", "linux", "ios", "android"}

def extract_cve_id(text: str) -> Optional[str]:
    match = CVE_RE.search(text or "")
    return match.group(0).upper() if match else None

def sanitize_phrase(text: str, *, max_words: int = 6, max_chars: int = 60) -> str:
    text = (text or "").strip()
    text = re.sub(r"[^A-Za-z0-9 ._\-/]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return ""
    words = text.split()[:max_words]
    text = " ".join(words)[:max_chars].strip()
    return text

def normalize_term(text: str) -> str:
    return sanitize_phrase(text, max_words=8, max_chars=80).lower()

def dedupe_preserve_order(items: list[str]) -> list[str]:
    seen = set()
    output: list[str] = []
    for item in items:
        if item and item not in seen:
            seen.add(item)
            output.append(item)
    return output

def normalize_platform(platform: str) -> str:
    value = normalize_term(platform)
    for canonical, aliases in PLATFORM_ALIASES.items():
        if value == canonical or value in aliases:
            return canonical
    return value

def find_platform_mentions(text: str) -> list[str]:
    lower = normalize_term(text)
    found: list[str] = []

    for canonical, aliases in PLATFORM_ALIASES.items():
        if canonical in lower or any(alias in lower for alias in aliases):
            found.append(canonical)

    for hint in COMMON_PLATFORM_HINTS:
        if hint in lower:
            found.append(hint)

    return dedupe_preserve_order(found)

def canonicalize_plan(plan: QueryPlan) -> QueryPlan:
    cve_id = extract_cve_id(plan.cve_id or "") if plan.cve_id else None

    required_concepts = [
        sanitize_phrase(term) for term in plan.required_concepts
        if sanitize_phrase(term)
    ]
    required_platforms = [
        normalize_platform(term) for term in plan.required_platforms
        if sanitize_phrase(term)
    ]
    optional_evidence_terms = [
        sanitize_phrase(term, max_words=4, max_chars=30) for term in plan.optional_evidence_terms
        if sanitize_phrase(term, max_words=4, max_chars=30)
    ]
    github_terms = [
        sanitize_phrase(term, max_words=6, max_chars=60) for term in plan.github_terms
        if sanitize_phrase(term, max_words=6, max_chars=60)
    ]

    required_concepts = dedupe_preserve_order(required_concepts)[:3]
    required_platforms = dedupe_preserve_order(required_platforms)[:3]
    optional_evidence_terms = dedupe_preserve_order(optional_evidence_terms)[:4]
    github_terms = dedupe_preserve_order(github_terms)[:5]

    if not required_concepts and cve_id:
        required_concepts = ["cve lookup"]

    if not optional_evidence_terms:
        optional_evidence_terms = ["exploit"]

    return QueryPlan(
        cve_id=cve_id,
        required_concepts=required_concepts or ["software vulnerability"],
        required_platforms=required_platforms,
        optional_evidence_terms=optional_evidence_terms,
        github_terms=github_terms,
        strict_platform=plan.strict_platform if required_platforms else False,
        reasoning=plan.reasoning,
    )

## Deterministic fallback planner

The notebook should still be teachable even if the LLM is off.

The deterministic planner is intentionally conservative:
it derives short semantic fields from the task and avoids inventing API syntax.


In [ ]:
def infer_platform_terms_from_task(task: str) -> list[str]:
    return find_platform_mentions(task)[:2]

def strip_platform_terms(text: str, platforms: list[str]) -> str:
    cleaned = text
    for platform in platforms:
        candidates = [platform] + PLATFORM_ALIASES.get(platform, [])
        for candidate in candidates:
            cleaned = re.sub(rf"\b{re.escape(candidate)}\b", " ", cleaned, flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", cleaned).strip()

def plan_query_deterministic(state: CollectorState) -> QueryPlan:
    task = state["task"]
    cve_id = extract_cve_id(task)

    if cve_id:
        return QueryPlan(
            cve_id=cve_id,
            required_concepts=["cve lookup"],
            required_platforms=[],
            optional_evidence_terms=["exploit"],
            github_terms=[cve_id],
            strict_platform=False,
            reasoning="The task already contains a CVE identifier, so use direct lookup first.",
        )

    required_platforms = infer_platform_terms_from_task(task)
    concept_seed = strip_platform_terms(task, required_platforms)
    concept_seed = sanitize_phrase(concept_seed, max_words=8, max_chars=80)

    github_terms = dedupe_preserve_order(
        [concept_seed] + required_platforms + ["exploit"]
    )

    return QueryPlan(
        cve_id=None,
        required_concepts=[concept_seed] if concept_seed else ["software vulnerability"],
        required_platforms=required_platforms,
        optional_evidence_terms=["exploit"],
        github_terms=[term for term in github_terms if term],
        strict_platform=bool(required_platforms),
        reasoning="Conservative fallback plan derived directly from the task text.",
    )

## LLM planner with `SystemMessage` and `HumanMessage`

The model does **not** produce raw API parameters.

Instead, it returns bounded semantic fields:

- **required concepts**,
- **required platforms or products**,
- **optional evidence terms**,
- **GitHub-friendly terms**,
- and an optional **CVE ID**.


In [ ]:
def plan_query_llm(state: CollectorState) -> QueryPlan:
    system_message = SystemMessage(
        content=(
            "You are planning a security collection workflow.\n"
            "Interpret the task into bounded semantic fields that another layer will turn into API calls.\n"
            "Do not return URLs, raw query parameters, or API syntax.\n"
            "Keep each phrase short, preserve important constraints from the task, and only include a platform or product when it is truly part of the task."
        )
    )

    human_message = HumanMessage(
        content=(
            f"Task: {state['task']}\n\n"
            "Return:\n"
            "- cve_id when a CVE is explicitly present\n"
            "- 1 to 3 required_concepts written as short phrases\n"
            "- required_platforms only when the task constrains the platform or product\n"
            "- a few optional_evidence_terms when they would help enrichment or validation\n"
            "- a few github_terms that could help find public repos or advisories\n"
            "- strict_platform=true only when the platform really matters for the task\n"
            "- short reasoning\n"
        )
    )

    raw_plan = call_llm_structured([system_message, human_message], QueryPlan)
    return canonicalize_plan(raw_plan)

def interpret_task_to_plan(state: CollectorState) -> QueryPlan:
    if USE_REAL_LLM and USE_STRUCTURED_OUTPUT:
        return plan_query_llm(state)
    return plan_query_deterministic(state)

## Query-building helpers

This is where Python turns the semantic plan into safe, ordered search candidates.

### NVD strategy

We intentionally put **strong conjunctive queries first**.

For example, for a task about **remote code execution on macOS**,
we want the early NVD queries to look more like:

- `remote code execution macOS`
- `Apple macOS remote code execution`
- exact phrase: `remote code execution`

Only later do we relax the search.

### GitHub strategy

GitHub is used in two ways:

1. **Enrichment** for accepted or partial NVD items,
2. **Task-level recovery** when NVD coverage is weak or platform coverage is missing.


In [ ]:

def is_safe_nvd_keyword_query(text: str) -> bool:
    return bool(text) and bool(SAFE_TERM_RE.fullmatch(text)) and len(text.split()) <= 8

def is_safe_github_query(text: str) -> bool:
    return bool(text) and bool(SAFE_GITHUB_QUERY_RE.fullmatch(text)) and len(text) <= 120

def expand_platform_aliases(platform: str) -> list[str]:
    canonical = normalize_platform(platform)
    aliases = PLATFORM_ALIASES.get(canonical, [canonical])
    return dedupe_preserve_order([sanitize_phrase(item, max_words=4, max_chars=30) for item in aliases])

def build_nvd_query_candidates(plan: QueryPlan) -> list[dict]:
    if plan.cve_id:
        return [{
            "kind": "cve_id",
            "label": f"Direct CVE lookup: {plan.cve_id}",
            "params": {"cveId": plan.cve_id},
            "display_query": plan.cve_id,
        }]

    candidates: list[dict] = []
    concepts = [sanitize_phrase(term, max_words=6, max_chars=60) for term in plan.required_concepts]
    concepts = [term for term in concepts if term]
    platforms = [normalize_platform(term) for term in plan.required_platforms]

    for concept in concepts:
        if len(concept.split()) >= 2:
            candidates.append({
                "kind": "keyword",
                "label": f"Exact concept phrase: {concept}",
                "params": {"keywordSearch": concept, "keywordExactMatch": "true"},
                "display_query": f'exact:{concept}',
            })

    for concept in concepts:
        for platform in platforms:
            base_platform_aliases = expand_platform_aliases(platform)
            primary_aliases = base_platform_aliases[:2]
            for alias in primary_aliases:
                query = sanitize_phrase(f"{concept} {alias}", max_words=8, max_chars=80)
                if is_safe_nvd_keyword_query(query):
                    candidates.append({
                        "kind": "keyword",
                        "label": f"Concept + platform: {query}",
                        "params": {"keywordSearch": query},
                        "display_query": query,
                    })

    for concept in concepts:
        if is_safe_nvd_keyword_query(concept):
            candidates.append({
                "kind": "keyword",
                "label": f"Concept only: {concept}",
                "params": {"keywordSearch": concept},
                "display_query": concept,
            })

    for concept in concepts:
        for platform in platforms:
            for evidence in plan.optional_evidence_terms[:2]:
                query = sanitize_phrase(f"{concept} {platform} {evidence}", max_words=8, max_chars=80)
                if is_safe_nvd_keyword_query(query):
                    candidates.append({
                        "kind": "keyword",
                        "label": f"Concept + platform + evidence: {query}",
                        "params": {"keywordSearch": query},
                        "display_query": query,
                    })

    output: list[dict] = []
    seen: set[str] = set()
    for item in candidates:
        key = json.dumps(item["params"], sort_keys=True)
        if key not in seen:
            seen.add(key)
            output.append(item)

    return output[:MAX_NVD_QUERIES]

def build_task_github_queries(plan: QueryPlan) -> list[str]:
    queries: list[str] = []

    if plan.cve_id:
        queries.extend([
            f'"{plan.cve_id}"',
            f'"{plan.cve_id}" exploit',
            f'"{plan.cve_id}" poc',
        ])

    concepts = [sanitize_phrase(term, max_words=6, max_chars=60) for term in plan.required_concepts if term]
    platforms = [normalize_platform(term) for term in plan.required_platforms if term]

    for concept in concepts:
        quoted_concept = f'"{concept}"' if " " in concept else concept
        if platforms:
            for platform in platforms[:2]:
                queries.append(f'{quoted_concept} {platform} in:name,description,readme')
                queries.append(f'{quoted_concept} {platform} exploit in:name,description,readme')
                queries.append(f'{quoted_concept} {platform} poc in:name,description,readme')
        else:
            queries.append(f'{quoted_concept} exploit in:name,description,readme')

    for term in plan.github_terms:
        cleaned = sanitize_phrase(term, max_words=6, max_chars=60)
        if cleaned:
            quoted = f'"{cleaned}"' if " " in cleaned else cleaned
            queries.append(f"{quoted} in:name,description,readme")

    output = [query for query in dedupe_preserve_order(queries) if is_safe_github_query(query)]
    return output[:MAX_GITHUB_RECOVERY_QUERIES]

def build_enrichment_queries(plan: QueryPlan, candidate: dict) -> list[str]:
    item_id = candidate.get("item_id", "")
    queries: list[str] = []

    if item_id and item_id.upper().startswith("CVE-"):
        queries.append(f'"{item_id}"')
        queries.append(f'"{item_id}" exploit')
        queries.append(f'"{item_id}" poc')
        for platform in plan.required_platforms[:2]:
            queries.append(f'"{item_id}" {platform}')

    for concept in plan.required_concepts[:2]:
        if item_id and item_id.upper().startswith("CVE-"):
            queries.append(f'"{item_id}" "{concept}"')
        for platform in plan.required_platforms[:1]:
            queries.append(f'"{concept}" {platform} exploit')

    output = [query for query in dedupe_preserve_order(queries) if is_safe_github_query(query)]
    return output[:4]


## Real tool layer

We use two public APIs:

- **NVD** as the main structured vulnerability source,
- **GitHub** as public no-auth enrichment and recovery.

The tool layer is plain Python.
That keeps it easy to inspect, test, and replace later.


In [ ]:


def extract_nvd_severity(cve: dict) -> Optional[str]:
    metrics = cve.get("metrics", {})
    for key in ("cvssMetricV40", "cvssMetricV31", "cvssMetricV30", "cvssMetricV2"):
        metric_list = metrics.get(key) or []
        if metric_list:
            metric = metric_list[0]
            if isinstance(metric, dict):
                cvss_data = metric.get("cvssData")
                if isinstance(cvss_data, dict):
                    severity = cvss_data.get("baseSeverity")
                    if severity:
                        return str(severity)
                severity = metric.get("baseSeverity")
                if severity:
                    return str(severity)
    return None

def fetch_nvd_cves_tool(query_spec: dict, results_per_page: int = NVD_RESULTS_PER_PAGE) -> list[dict]:
    url = "https://services.nvd.nist.gov/rest/json/cves/2.0"
    headers = {
        "User-Agent": USER_AGENT,
        "Accept": "application/json",
    }

    params = dict(query_spec.get("params", {}))
    params.setdefault("resultsPerPage", results_per_page)

    response = requests.get(url, headers=headers, params=params, timeout=30)
    if response.status_code >= 400:
        message = response.headers.get("message", "")
        detail = f"NVD request failed with status {response.status_code}"
        if message:
            detail += f": {message}"
        raise RuntimeError(detail)

    data = response.json()
    items: list[dict] = []

    for vuln in data.get("vulnerabilities", []):
        cve = vuln.get("cve", {})
        cve_id = cve.get("id", "")

        descriptions = cve.get("descriptions", []) or []
        description = next(
            (entry.get("value", "") for entry in descriptions if entry.get("lang") == "en"),
            "",
        )

        items.append({
            "source": "nvd",
            "item_id": cve_id,
            "url": f"https://nvd.nist.gov/vuln/detail/{cve_id}" if cve_id else None,
            "text_preview": description[:1000],
            "severity_hint": extract_nvd_severity(cve),
        })

    return items

def fetch_github_repos_tool(query: str, per_page: int = GITHUB_RESULTS_PER_PAGE) -> list[dict]:
    url = "https://api.github.com/search/repositories"
    headers = {
        "Accept": "application/vnd.github+json",
        "User-Agent": USER_AGENT,
    }
    params = {
        "q": query,
        "sort": "stars",
        "order": "desc",
        "per_page": per_page,
    }

    response = requests.get(url, headers=headers, params=params, timeout=30)

    if response.status_code in (403, 429):
        remaining = response.headers.get("x-ratelimit-remaining")
        reset_at = response.headers.get("x-ratelimit-reset")
        retry_after = response.headers.get("retry-after")
        detail = f"GitHub search limited with status {response.status_code}."
        if remaining is not None:
            detail += f" remaining={remaining}."
        if retry_after:
            detail += f" retry_after={retry_after}s."
        if reset_at:
            detail += f" reset={reset_at}."
        raise RuntimeError(detail)

    if response.status_code == 422:
        raise RuntimeError(f"GitHub rejected the search query as invalid: {query}")

    response.raise_for_status()
    data = response.json()

    repos: list[dict] = []
    for item in data.get("items", []):
        name = item.get("full_name", "")
        description = item.get("description", "") or ""
        html_url = item.get("html_url", "")
        preview = f"{name}. {description}".strip()

        repos.append({
            "source": "github",
            "item_id": name or html_url or "unknown-repo",
            "url": html_url,
            "text_preview": preview[:500],
            "severity_hint": None,
        })

    return repos


## Validation and extraction helpers

It does **not** accept an item merely because it is a structured vulnerability record.

Instead, it scores:

- **required concept matches**,
- **platform matches**,
- **evidence terms**,
- and possible **platform conflicts**.

This lets the graph distinguish:

- **strong matches**,
- **partial matches**,
- and **rejections**.


In [ ]:
def phrase_acronym(text: str) -> str:
    tokens = re.findall(r"[A-Za-z0-9]+", text.lower())
    if len(tokens) < 2:
        return ""
    return "".join(token[0] for token in tokens if token)

def concept_tokens(text: str) -> list[str]:
    return [token for token in re.findall(r"[A-Za-z0-9]+", text.lower()) if len(token) >= 3]

def text_matches_concept(text: str, concept: str) -> list[str]:
    lower = text.lower()
    normalized_concept = normalize_term(concept)
    matches: list[str] = []

    if normalized_concept and normalized_concept in lower:
        matches.append(normalized_concept)

    acronym = phrase_acronym(concept)
    if acronym and re.search(rf"\b{re.escape(acronym)}\b", lower):
        matches.append(acronym)

    tokens = concept_tokens(concept)
    if tokens:
        present = [token for token in tokens if re.search(rf"\b{re.escape(token)}\b", lower)]
        min_needed = len(tokens) if len(tokens) <= 2 else max(2, len(tokens) - 1)
        if len(present) >= min_needed:
            matches.extend(present)

    return dedupe_preserve_order(matches)

def matched_platforms_in_text(text: str, required_platforms: list[str]) -> list[str]:
    present = find_platform_mentions(text)
    normalized_required = {normalize_platform(item) for item in required_platforms}
    return [item for item in present if item in normalized_required]

def validate_candidate_against_plan(candidate: dict, plan: QueryPlan) -> ValidationDecision:
    text = (candidate.get("text_preview") or "").lower()
    source = candidate.get("source", "nvd")

    score = 0
    matched_concepts: list[str] = []
    matched_platforms: list[str] = []

    for concept in plan.required_concepts:
        found = text_matches_concept(text, concept)
        if found:
            matched_concepts.append(concept)
            if normalize_term(concept) in text:
                score += 3
            else:
                score += 2

    matched_platforms = matched_platforms_in_text(text, plan.required_platforms)
    score += 2 * len(matched_platforms)

    for evidence_term in plan.optional_evidence_terms:
        if evidence_term.lower() in text:
            score += 1

    if source == "nvd":
        score += 1

    if plan.required_platforms and not matched_platforms:
        visible_platforms = set(find_platform_mentions(text))
        required_platform_set = {normalize_platform(item) for item in plan.required_platforms}
        conflicting = visible_platforms - required_platform_set
        if conflicting:
            score -= 2

    has_concept_match = bool(matched_concepts)
    has_platform_match = bool(matched_platforms)

    accepted_threshold = 5 if plan.required_platforms and plan.strict_platform else 4
    accepted = has_concept_match and score >= accepted_threshold and (has_platform_match or not plan.strict_platform)
    partial_match = has_concept_match and not accepted

    if accepted:
        reason = (
            f"strong match with score={score}; "
            f"concepts={matched_concepts or ['none']}; "
            f"platforms={matched_platforms or ['none']}"
        )
    elif partial_match:
        reason = (
            f"partial match with score={score}; "
            f"concepts={matched_concepts or ['none']}; "
            f"platforms={matched_platforms or ['none']}"
        )
    else:
        reason = (
            f"weak match with score={score}; "
            f"concepts={matched_concepts or ['none']}; "
            f"platforms={matched_platforms or ['none']}"
        )

    return ValidationDecision(
        accepted=accepted,
        partial_match=partial_match,
        score=score,
        matched_concepts=matched_concepts,
        matched_platforms=matched_platforms,
        reason=reason,
    )

def build_candidate_record(candidate: dict, decision: ValidationDecision) -> CandidateRecord:
    return CandidateRecord(
        source=candidate.get("source", "nvd"),
        item_id=candidate.get("item_id", "unknown-item"),
        url=candidate.get("url"),
        text_preview=(candidate.get("text_preview") or "")[:250],
        accepted=decision.accepted,
        partial_match=decision.partial_match,
        score=decision.score,
        matched_concepts=decision.matched_concepts,
        matched_platforms=decision.matched_platforms,
        reason=decision.reason,
    )

def extract_signals_from_candidate(candidate: dict, plan: QueryPlan) -> ExtractedSignals:
    text = (candidate.get("text_preview") or "").lower()

    matched_terms: list[str] = []
    for concept in plan.required_concepts:
        if text_matches_concept(text, concept):
            matched_terms.append(concept)

    platform_terms_found = matched_platforms_in_text(text, plan.required_platforms)

    return ExtractedSignals(
        source=candidate.get("source", "unknown"),
        item_id=candidate.get("item_id", "unknown-item"),
        matched_terms=dedupe_preserve_order(matched_terms),
        platform_terms_found=dedupe_preserve_order(platform_terms_found),
        mentions_exploit=("exploit" in text) or ("proof of concept" in text) or ("poc" in text),
        mentions_remote_code_execution=bool(text_matches_concept(text, "remote code execution")),
        mentions_prompt_injection=bool(text_matches_concept(text, "prompt injection")),
        mentions_data_exfiltration=bool(text_matches_concept(text, "data exfiltration")),
        mentions_privilege_escalation=bool(text_matches_concept(text, "privilege escalation")),
        severity_hint=candidate.get("severity_hint"),
    )

def compute_coverage(state: CollectorState, plan: QueryPlan) -> dict:
    strong_items = state["accepted_items"]
    strong_count = len(strong_items)
    strong_platform_count = sum(1 for item in strong_items if item.get("matched_platforms"))
    partial_count = len(state["partial_items"])
    github_enrichment_count = len(state["github_enrichments"])

    needs_platform = bool(plan.required_platforms) and plan.strict_platform
    has_platform_coverage = (strong_platform_count >= TARGET_PLATFORM_STRONG_MATCHES) if needs_platform else True
    enough_strong_items = strong_count >= TARGET_STRONG_MATCHES

    return {
        "strong_count": strong_count,
        "strong_platform_count": strong_platform_count,
        "partial_count": partial_count,
        "github_enrichment_count": github_enrichment_count,
        "needs_platform": needs_platform,
        "has_platform_coverage": has_platform_coverage,
        "enough_strong_items": enough_strong_items,
    }

## Node functions

Each node does one clear thing.

This notebook deliberately keeps the control flow visible instead of hiding behavior behind a high-level agent abstraction.


In [ ]:

def initialize_collection(state: CollectorState) -> dict:
    return {
        "interpreted_plan": None,
        "search_phase": "nvd",
        "nvd_query_candidates": [],
        "nvd_query_index": 0,
        "current_nvd_query": None,
        "github_recovery_queries": [],
        "github_recovery_query_index": 0,
        "current_github_recovery_query": None,
        "candidate_items": [],
        "current_candidate_index": 0,
        "current_candidate": None,
        "accepted_items": [],
        "partial_items": [],
        "rejected_items": [],
        "github_enrichments": [],
        "extracted_signals": [],
        "queries_tried": [],
        "run_log": ["Initialized collection workflow."],
        "errors": [],
        "iteration": 0,
        "accepted_count_at_last_assessment": 0,
        "no_new_accept_rounds": 0,
        "last_validation_accepted": False,
        "last_validation_partial": False,
        "last_validation_source": None,
        "github_recovery_attempted": False,
        "stop_reason": None,
        "summary": None,
    }

def interpret_task(state: CollectorState) -> dict:
    plan = interpret_task_to_plan(state)
    return {
        "interpreted_plan": plan.model_dump(),
        "run_log": state["run_log"] + [f"Interpreted task into structured plan: {plan.reasoning}"],
    }

def build_query_candidates(state: CollectorState) -> dict:
    plan = QueryPlan(**(state["interpreted_plan"] or {}))
    nvd_queries = build_nvd_query_candidates(plan)
    github_queries = build_task_github_queries(plan)

    nvd_display = [item["display_query"] for item in nvd_queries]
    return {
        "nvd_query_candidates": nvd_queries,
        "github_recovery_queries": github_queries,
        "run_log": state["run_log"] + [
            f"Built {len(nvd_queries)} NVD query candidate(s): {nvd_display}",
            f"Built {len(github_queries)} GitHub recovery query candidate(s): {github_queries}",
        ],
    }

def prepare_next_query(state: CollectorState) -> dict:
    phase = state["search_phase"]

    if phase == "nvd":
        index = state["nvd_query_index"]
        queries = state["nvd_query_candidates"]
        if index >= len(queries):
            return {
                "current_nvd_query": None,
                "run_log": state["run_log"] + ["No remaining NVD queries."],
            }
        current = queries[index]
        return {
            "current_nvd_query": current,
            "nvd_query_index": index + 1,
            "iteration": state["iteration"] + 1,
            "queries_tried": state["queries_tried"] + [f"nvd:{current['display_query']}"],
            "run_log": state["run_log"] + [f"Prepared NVD query: {current['display_query']}"],
        }

    index = state["github_recovery_query_index"]
    queries = state["github_recovery_queries"]
    if index >= len(queries):
        return {
            "current_github_recovery_query": None,
            "run_log": state["run_log"] + ["No remaining GitHub recovery queries."],
        }

    current = queries[index]
    return {
        "current_github_recovery_query": current,
        "github_recovery_query_index": index + 1,
        "iteration": state["iteration"] + 1,
        "queries_tried": state["queries_tried"] + [f"github:{current}"],
        "run_log": state["run_log"] + [f"Prepared GitHub recovery query: {current}"],
    }

def fetch_candidates(state: CollectorState) -> dict:
    phase = state["search_phase"]
    errors = list(state["errors"])
    run_log = list(state["run_log"])

    try:
        if phase == "nvd":
            spec = state["current_nvd_query"]
            if not spec:
                return {
                    "candidate_items": [],
                    "current_candidate_index": 0,
                    "current_candidate": None,
                    "run_log": run_log + ["Skipped NVD fetch because there was no current NVD query."],
                }

            items = fetch_nvd_cves_tool(spec) if USE_REAL_TOOLS else []
            run_log.append(f"Fetched {len(items)} NVD candidate(s) for query: {spec['display_query']}")
            return {
                "candidate_items": items,
                "current_candidate_index": 0,
                "current_candidate": None,
                "run_log": run_log,
                "errors": errors,
            }

        query = state["current_github_recovery_query"]
        if not query:
            return {
                "candidate_items": [],
                "current_candidate_index": 0,
                "current_candidate": None,
                "run_log": run_log + ["Skipped GitHub recovery fetch because there was no current GitHub query."],
            }

        items = fetch_github_repos_tool(query) if USE_REAL_TOOLS else []
        run_log.append(f"Fetched {len(items)} GitHub recovery candidate(s) for query: {query}")
        return {
            "candidate_items": items,
            "current_candidate_index": 0,
            "current_candidate": None,
            "run_log": run_log,
            "errors": errors,
        }

    except Exception as exc:
        errors.append(str(exc))
        if phase == "nvd":
            run_log.append(f"NVD fetch failed: {exc}")
        else:
            run_log.append(f"GitHub recovery fetch failed: {exc}")

        return {
            "candidate_items": [],
            "current_candidate_index": 0,
            "current_candidate": None,
            "errors": errors,
            "run_log": run_log,
        }

def pick_candidate(state: CollectorState) -> dict:
    idx = state["current_candidate_index"]
    items = state["candidate_items"]

    if idx >= len(items):
        return {
            "current_candidate": None,
            "run_log": state["run_log"] + ["No more candidates in the current batch."],
        }

    candidate = items[idx]
    return {
        "current_candidate": candidate,
        "current_candidate_index": idx + 1,
        "run_log": state["run_log"] + [
            f"Selected candidate {idx + 1}/{len(items)} from {candidate.get('source', 'unknown')}: {candidate.get('item_id', 'unknown-item')}"
        ],
    }


In [ ]:

def validate_candidate(state: CollectorState) -> dict:
    candidate = state.get("current_candidate")
    if not candidate:
        return {
            "last_validation_accepted": False,
            "last_validation_partial": False,
            "last_validation_source": None,
            "run_log": state["run_log"] + ["Validation skipped because there was no current candidate."],
        }

    plan = QueryPlan(**(state["interpreted_plan"] or {}))
    decision = validate_candidate_against_plan(candidate, plan)
    record = build_candidate_record(candidate, decision).model_dump()

    updates = {
        "last_validation_accepted": decision.accepted,
        "last_validation_partial": decision.partial_match,
        "last_validation_source": candidate.get("source"),
    }

    if decision.accepted:
        return {
            **updates,
            "accepted_items": state["accepted_items"] + [record],
            "run_log": state["run_log"] + [f"Accepted candidate: {record['item_id']} ({record['reason']})"],
        }

    if decision.partial_match:
        return {
            **updates,
            "partial_items": state["partial_items"] + [record],
            "run_log": state["run_log"] + [f"Stored partial match: {record['item_id']} ({record['reason']})"],
        }

    return {
        **updates,
        "rejected_items": state["rejected_items"] + [record],
        "run_log": state["run_log"] + [f"Rejected candidate: {record['item_id']} ({record['reason']})"],
    }

def enrich_with_github(state: CollectorState) -> dict:
    candidate = state.get("current_candidate")
    if not candidate:
        return {
            "run_log": state["run_log"] + ["GitHub enrichment skipped because there was no current candidate."],
        }

    if candidate.get("source") != "nvd" or not ENABLE_GITHUB_ENRICHMENT:
        return {
            "run_log": state["run_log"] + [f"GitHub enrichment skipped for source={candidate.get('source')}."],
        }

    plan = QueryPlan(**(state["interpreted_plan"] or {}))
    queries = build_enrichment_queries(plan, candidate)
    if not queries:
        return {
            "run_log": state["run_log"] + [f"GitHub enrichment found no safe query candidates for {candidate.get('item_id')}."],
        }

    enrichments: list[dict] = []
    errors = list(state["errors"])
    run_log = list(state["run_log"])

    for query in queries[:2]:
        try:
            repos = fetch_github_repos_tool(query) if USE_REAL_TOOLS else []
            for repo in repos[:2]:
                enrichments.append(
                    EnrichmentRecord(
                        related_to_item_id=candidate.get("item_id", "unknown-item"),
                        query=query,
                        repo_name=repo.get("item_id", "unknown-repo"),
                        repo_url=repo.get("url") or "",
                        text_preview=repo.get("text_preview", "")[:250],
                    ).model_dump()
                )
            run_log.append(
                f"Fetched {len(repos)} GitHub enrichment repo(s) for {candidate.get('item_id')} using query: {query}"
            )
        except Exception as exc:
            errors.append(str(exc))
            run_log.append(f"GitHub enrichment failed for query '{query}': {exc}")

    return {
        "github_enrichments": state["github_enrichments"] + enrichments,
        "errors": errors,
        "run_log": run_log,
    }

def extract_signals(state: CollectorState) -> dict:
    candidate = state.get("current_candidate")
    if not candidate:
        return {
            "run_log": state["run_log"] + ["Signal extraction skipped because there was no current candidate."],
        }

    plan = QueryPlan(**(state["interpreted_plan"] or {}))
    signals = extract_signals_from_candidate(candidate, plan)

    return {
        "extracted_signals": state["extracted_signals"] + [signals.model_dump()],
        "run_log": state["run_log"] + [f"Extracted signals for: {candidate.get('item_id', 'unknown-item')}"],
    }

def assess_progress(state: CollectorState) -> dict:
    plan = QueryPlan(**(state["interpreted_plan"] or {}))
    coverage = compute_coverage(state, plan)

    current_strong = coverage["strong_count"]
    previous_strong = state["accepted_count_at_last_assessment"]
    if current_strong > previous_strong:
        stale_rounds = 0
    else:
        stale_rounds = state["no_new_accept_rounds"] + 1

    more_nvd_queries = state["nvd_query_index"] < len(state["nvd_query_candidates"])
    more_github_queries = state["github_recovery_query_index"] < len(state["github_recovery_queries"])

    enough_for_stop = (
        coverage["enough_strong_items"]
        and coverage["has_platform_coverage"]
        and (
            not ENABLE_GITHUB_ENRICHMENT
            or coverage["github_enrichment_count"] > 0
            or not plan.required_platforms
        )
    )

    stop_reason = None
    if enough_for_stop:
        stop_reason = "Task coverage satisfied with strong matches."
    elif state["iteration"] >= state["max_iterations"]:
        stop_reason = "Maximum query rounds reached."
    elif stale_rounds >= state["max_stale_rounds"] and not more_nvd_queries and not more_github_queries:
        stop_reason = "Search became stale and no better queries remain."
    elif state["search_phase"] == "github_recovery" and not more_github_queries:
        stop_reason = "GitHub recovery queries exhausted."
    elif state["search_phase"] == "nvd" and not more_nvd_queries and not ENABLE_TASK_GITHUB_RECOVERY:
        stop_reason = "NVD queries exhausted."

    return {
        "accepted_count_at_last_assessment": current_strong,
        "no_new_accept_rounds": stale_rounds,
        "stop_reason": stop_reason,
        "run_log": state["run_log"] + [
            "Assessed progress. "
            f"strong={coverage['strong_count']}, "
            f"platform_strong={coverage['strong_platform_count']}, "
            f"partials={coverage['partial_count']}, "
            f"github_enrichments={coverage['github_enrichment_count']}, "
            f"stale_rounds={stale_rounds}, "
            f"stop_reason={stop_reason}"
        ],
    }

def switch_to_github_recovery(state: CollectorState) -> dict:
    return {
        "search_phase": "github_recovery",
        "github_recovery_attempted": True,
        "run_log": state["run_log"] + ["Switching to task-level GitHub recovery because NVD coverage is still weak."],
    }

def finalize_collection(state: CollectorState) -> dict:
    plan = QueryPlan(**(state["interpreted_plan"] or {}))
    coverage = compute_coverage(state, plan)

    if USE_REAL_LLM:
        messages = [
            SystemMessage(
                content=(
                    "You are summarizing a workshop collection run. "
                    "Explain what the agent found, how strong the match was, and whether the task constraints were fully satisfied."
                )
            ),
            HumanMessage(
                content=(
                    f"Task: {state['task']}\n\n"
                    f"Plan: {json.dumps(state['interpreted_plan'], indent=2)}\n\n"
                    f"Accepted items: {json.dumps(state['accepted_items'], indent=2)}\n\n"
                    f"Partial items: {json.dumps(state['partial_items'], indent=2)}\n\n"
                    f"GitHub enrichments: {json.dumps(state['github_enrichments'], indent=2)}\n\n"
                    f"Coverage: {json.dumps(coverage, indent=2)}\n\n"
                    f"Stop reason: {state['stop_reason']}"
                )
            ),
        ]
        summary = call_llm_text(messages)
    else:
        summary = (
            f"Finished with {coverage['strong_count']} strong item(s), "
            f"{coverage['partial_count']} partial item(s), and "
            f"{coverage['github_enrichment_count']} GitHub enrichment record(s). "
            f"Stop reason: {state['stop_reason']}"
        )

    return {
        "summary": summary,
        "run_log": state["run_log"] + ["Finalized collection workflow."],
    }


## Route helpers

The route functions are intentionally small and readable.


In [ ]:

def route_after_pick(state: CollectorState) -> str:
    if state["current_candidate"] is None:
        return "assess_progress"
    return "validate_candidate"

def route_after_validation(state: CollectorState) -> str:
    if state["last_validation_accepted"] or state["last_validation_partial"]:
        return "enrich_with_github"
    return "pick_candidate"

def route_after_assessment(state: CollectorState) -> str:
    if state["stop_reason"] is not None:
        return "finalize"

    plan = QueryPlan(**(state["interpreted_plan"] or {}))
    coverage = compute_coverage(state, plan)
    more_nvd_queries = state["nvd_query_index"] < len(state["nvd_query_candidates"])
    more_github_queries = state["github_recovery_query_index"] < len(state["github_recovery_queries"])

    if state["search_phase"] == "nvd":
        need_platform_recovery = coverage["needs_platform"] and not coverage["has_platform_coverage"]
        need_better_public_evidence = coverage["strong_count"] < TARGET_STRONG_MATCHES
        github_has_not_helped_yet = ENABLE_GITHUB_ENRICHMENT and len(state["github_enrichments"]) == 0

        if more_nvd_queries and (need_platform_recovery or need_better_public_evidence or github_has_not_helped_yet):
            return "prepare_next_query"

        if ENABLE_TASK_GITHUB_RECOVERY and more_github_queries and (need_platform_recovery or need_better_public_evidence or coverage["partial_count"] > 0):
            return "switch_to_github_recovery"

        return "finalize"

    if more_github_queries:
        return "prepare_next_query"

    return "finalize"


## Build the graph

Read the graph aloud as a workflow:

1. initialize,
2. interpret the task,
3. build safe query candidates,
4. fetch one batch,
5. validate candidates one by one,
6. enrich accepted or partial matches,
7. assess whether the task is actually satisfied,
8. continue, recover, or stop.


In [ ]:

graph_builder = StateGraph(CollectorState)

graph_builder.add_node("initialize", initialize_collection)
graph_builder.add_node("interpret_task", interpret_task)
graph_builder.add_node("build_query_candidates", build_query_candidates)
graph_builder.add_node("prepare_next_query", prepare_next_query)
graph_builder.add_node("fetch_candidates", fetch_candidates)
graph_builder.add_node("pick_candidate", pick_candidate)
graph_builder.add_node("validate_candidate", validate_candidate)
graph_builder.add_node("enrich_with_github", enrich_with_github)
graph_builder.add_node("extract_signals", extract_signals)
graph_builder.add_node("assess_progress", assess_progress)
graph_builder.add_node("switch_to_github_recovery", switch_to_github_recovery)
graph_builder.add_node("finalize", finalize_collection)

graph_builder.add_edge(START, "initialize")
graph_builder.add_edge("initialize", "interpret_task")
graph_builder.add_edge("interpret_task", "build_query_candidates")
graph_builder.add_edge("build_query_candidates", "prepare_next_query")
graph_builder.add_edge("prepare_next_query", "fetch_candidates")
graph_builder.add_edge("fetch_candidates", "pick_candidate")

graph_builder.add_conditional_edges(
    "pick_candidate",
    route_after_pick,
    {
        "validate_candidate": "validate_candidate",
        "assess_progress": "assess_progress",
    },
)

graph_builder.add_conditional_edges(
    "validate_candidate",
    route_after_validation,
    {
        "enrich_with_github": "enrich_with_github",
        "pick_candidate": "pick_candidate",
    },
)

graph_builder.add_edge("enrich_with_github", "extract_signals")
graph_builder.add_edge("extract_signals", "pick_candidate")

graph_builder.add_conditional_edges(
    "assess_progress",
    route_after_assessment,
    {
        "prepare_next_query": "prepare_next_query",
        "switch_to_github_recovery": "switch_to_github_recovery",
        "finalize": "finalize",
    },
)

graph_builder.add_edge("switch_to_github_recovery", "prepare_next_query")
graph_builder.add_edge("finalize", END)

collector_graph = graph_builder.compile(
    checkpointer=checkpointer if USE_CHECKPOINTER else None
)


## Run the graph with `invoke()`

The example task below is deliberately the one that exposed the weakness in the earlier version.

This stricter notebook should:

- prioritize a stronger NVD query,
- resist weak matches,
- keep searching if macOS coverage is missing,
- and use GitHub more meaningfully when recovery is needed.


In [ ]:

initial_state: CollectorState = {
    "task": "Collect public vulnerability evidence related to remote code execution on macOS.",
    "interpreted_plan": None,
    "search_phase": "nvd",
    "nvd_query_candidates": [],
    "nvd_query_index": 0,
    "current_nvd_query": None,
    "github_recovery_queries": [],
    "github_recovery_query_index": 0,
    "current_github_recovery_query": None,
    "candidate_items": [],
    "current_candidate_index": 0,
    "current_candidate": None,
    "accepted_items": [],
    "partial_items": [],
    "rejected_items": [],
    "github_enrichments": [],
    "extracted_signals": [],
    "queries_tried": [],
    "run_log": [],
    "errors": [],
    "iteration": 0,
    "max_iterations": 6,
    "accepted_count_at_last_assessment": 0,
    "no_new_accept_rounds": 0,
    "max_stale_rounds": 2,
    "last_validation_accepted": False,
    "last_validation_partial": False,
    "last_validation_source": None,
    "github_recovery_attempted": False,
    "stop_reason": None,
    "summary": None,
}

result = collector_graph.invoke(initial_state)
result


## Inspect the run

In [ ]:

print("Queries tried:")
for item in result["queries_tried"]:
    print("-", item)

print("\nAccepted items:")
for item in result["accepted_items"]:
    print("-", item["item_id"], "| score:", item["score"], "| platforms:", item["matched_platforms"], "| reason:", item["reason"])

print("\nPartial items:")
for item in result["partial_items"]:
    print("-", item["item_id"], "| score:", item["score"], "| reason:", item["reason"])

print("\nGitHub enrichments:")
for item in result["github_enrichments"][:10]:
    print("-", item["related_to_item_id"], "->", item["repo_name"], "| query:", item["query"])

print("\nStop reason:")
print(result["stop_reason"])

print("\nSummary:")
print(result["summary"])


## Inspect the run log

Observability is part of the design, not an afterthought.


In [ ]:

for entry in result["run_log"]:
    print(entry)


## Save the result

Persisting runs is important because the exercise is about iterative systems that collect, compare, and improve over time.


In [ ]:

def save_run(result: dict, path: str = "collector_run_output.json") -> None:
    Path(path).write_text(json.dumps(result, indent=2))


## Why this version maps better to the course exercise

This graph is closer to the real implementations because it explicitly separates:

- **task interpretation**
- **query construction**
- **real-source collection**
- **validation**
- **enrichment**
- **reflection**
- **stopping logic**
- **persistence**

The important lesson is that an agent is a **stateful iterative workflow**,
not just a chatbot that occasionally calls tools.
